In [1]:
from agents import Agent, Production, Chat, Toolkit, Prompt
from pydantic import BaseModel

### **Toolkit**

Toolkits are MCP servers that are externally run. Before initializing a Toolkit object, the MCP server needs to be operational.

In [2]:
utils_toolkit = Toolkit(name = 'Utilities', url = 'http://localhost:9001/mcp')
iris_toolkit = Toolkit(name='IRIS', url = 'http://localhost:9002/mcp')

### **Chat**

The Chat API can be used to persist conversations. A chat id can be used to construct a history of that Chat from IRIS instead of needing to maintain it manually. This is particularly important when Enterprise licenses for OpenAI have Zero Data Retention enabled and so OpenAI is not authorized to store the conversation on their servers, the Chat API allows for constructing the conversation from history stored in IRIS.

In [3]:
context = Chat(
    name="travel",
    messages=[
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "We are in Washington DC"},
        {"role": "assistant", "content": "Great, what do you want to do in DC?"}
    ]
)
context

Chat(name='travel', messages=3)

In [4]:
context.messages

[{'role': 'system', 'content': 'You are helpful.'},
 {'role': 'user', 'content': 'We are in Washington DC'},
 {'role': 'assistant', 'content': 'Great, what do you want to do in DC?'}]

In [5]:
context == Chat('travel')

True

### **Prompt**

- The Prompt API is a way to manage and version Prompts. 
- Prompts can be built at runtime using parameters. 
- Prompts Prompts versions can be fetched by a selected version. 
- Variables contained in a prompt can be queried using `get_variables()` method.

In [6]:
bond_system = Prompt(name='Agent007', text='You are {agent_name}. You always stay in character.')
bond_system.build(agent_name='James Bond')

'You are James Bond. You always stay in character.'

In [7]:
bond_system = Prompt(name='Agent007', text='Your next mission is of utmost importance, you do not have time to talk.')
bond_system

Prompt(name='Agent007', version=2, text='Your next mission is of utmost importance, you do not have time to talk.')

In [8]:
Prompt('Agent007') == bond_system

True

In [9]:
Prompt('Agent007', version=1)

Prompt(name='Agent007', version=1, text='You are {agent_name}. You always stay in character.')

In [10]:
Prompt('Agent007', version=1).get_variables()

['agent_name']

In [11]:
Prompt('Agent007').delete()
try:
    prompt = Prompt('Agent007')
except ValueError as e:
    print(e)

No prompt found for 'Agent007'


### **Agents**

Agents can be defined by a name, a description (not currently used in any way but can be leveraged in the future for expert selection), and an OpenAI model. Optionally, agents can be configured with a default structured output (modifiable at call time) and a set of toolkits the agent should have access to. These tools are advertised to the LLM specific to access the agent has at a Toolkit level (specifying individual tools inside a Toolkit is not currently supported). Agents must be added to a Production before being used.

In [12]:
molly = Agent(name='Molly', model='gpt-5')
Production('AgentSpace', [molly]).start()
molly('What are some summer hiking trails around Boston?')


Load started on 04/16/2026 16:01:58
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/16/2026 16:01:58
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/16/2026 16:01:59
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/16/2026 16:01:59
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/16/2026 16:01:59
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

'Here are great summer-friendly hiking options around Boston (with transit notes where doable):\n\n- Blue Hills Reservation (Milton/Quincy): Skyline Trail 7-10 mi with rocky ups/downs and views from Great Blue Hill; many shorter loops. MBTA buses 238/240 reach trailheads; weekend parking fills early.\n\n- Middlesex Fells Reservation (Medford/Stoneham/Winchester): Skyline 7.5 mi ridge loop; Rock Circuit ~4 mi; lots of shaded woods and ponds. Orange Line to Oak Grove or buses to various entrances.\n\n- Walden Pond State Reservation (Concord): Easy 1.7 mi pond loop plus links to Concord conservation lands (3-6+ mi). Fitchburg Line to Concord or Lincoln, then ~1 mi walk; swimming area gets busy.\n\n- Minute Man National Historical Park (Lexington/Concord): Battle Road Trail ~5 mi one-way on crushed gravel; historic sites and shade. Red Line to Alewife + bus, or commuter rail to Concord.\n\n- Lynn Woods Reservation (Lynn): Stone Tower, Weetamoo Cliff, Dungeon Rock; mix of carriage roads and

Agents can be fetched using only their name. Adding any other parameters will be treated as agent creation.

In [13]:
Agent('Molly') == molly

True

Agents can be configured with a default reasoning effort and response format, but these parameters can be overridden at runtime.

In [14]:
class AlexResponse(BaseModel):
    message: str
    reasoning: str

class MollyResponse(BaseModel):
    text: str
    reasoning: str

alex = Agent(name='Alex', 
             description='Test Agent 1', 
             system_prompt=Prompt(name='alex_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit],
             response_format=AlexResponse)

molly = Agent(name='Molly', 
             description='Test Agent 2', 
             system_prompt=Prompt(name='molly_system', text='You are a helpful agent'),
             model='gpt-5',
             reasoning_effort='low',
             toolkits=[utils_toolkit, iris_toolkit],
             response_format=MollyResponse)


Load started on 04/16/2026 16:02:41
Loading file Agents.Message.AlexResponse.cls as udl
Compiling class Agents.Message.AlexResponse
Compiling table Agents_Message.AlexResponse
Compiling routine Agents.Message.AlexResponse.1
Load finished successfully.

Load started on 04/16/2026 16:02:42
Loading file Agents.Message.MollyResponse.cls as udl
Compiling class Agents.Message.MollyResponse
Compiling table Agents_Message.MollyResponse
Compiling routine Agents.Message.MollyResponse.1
Load finished successfully.


In [15]:
Production('AgentSpace', [molly, alex]).start()


Load started on 04/16/2026 16:02:43
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/16/2026 16:02:43
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/16/2026 16:02:44
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/16/2026 16:02:44
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/16/2026 16:02:44
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

Without any context, the agent does not construct its memory from the database, nor does it log the response, operating in a stateless manner. Model usage is still logged.

In [16]:
molly(message='Which tables do we have in IRIS in the Agents namespace?')

'{"text": "The Agents namespace has these tables: SQLUser.Agent, SQLUser.AgentToolkit, SQLUser.Chat, SQLUser.Prompt, SQLUser.TestModel, SQLUser.Toolkit, SQLUser.ToolUsage, SQLUser.Usage.", "reasoning": "Used the provided IRIS list_tables tool result to extract the tables."}'

Agents can be directly invoked to use a tool using compatible parameters

In [17]:
molly.use(toolkit='IRIS', tool='query', params={'namespace':'Agents', 'sql': 'Select * from Agent'})

[{'agent_name': 'Alex',
  'description': 'Test Agent 1',
  'system_prompt_id': 'alex_system',
  'model': 'gpt-5',
  'response_format': 'AlexResponse',
  'reasoning_effort': 'medium',
  'persist_reasoning': '1'},
 {'agent_name': 'Molly',
  'description': 'Test Agent 2',
  'system_prompt_id': 'molly_system',
  'model': 'gpt-5',
  'response_format': 'MollyResponse',
  'reasoning_effort': 'low',
  'persist_reasoning': '1'}]

When context is specified, the agent constructs its conversational context from database before querying the LLM

In [18]:
molly(message='What is the weather today?', chat=context)

'{"text": "Today\'s weather in Washington, DC: Cloudy, high 26\\u00b0, low 13\\u00b0.", "reasoning": "Used the previously returned Utilities.weather result for Washington, DC."}'

Here, we override the agent's default reasoning effort at runtime

In [19]:
molly(message='Recommend some good food spots for lunch', chat='travel', reasoning_effort='high')

'{"text": "Here are solid lunch picks around DC:\\n\\nQuick/casual\\n- RASA (Indian bowls; multiple locations)\\n- Little Sesame (hummus + pita; multiple locations)\\n- Shouk (plant-based Israeli; multiple locations)\\n- Falafel Inc (cheap, fast; multiple locations incl. Georgetown)\\n- Chaia Tacos (vegetarian tacos; Georgetown and more)\\n- &pizza or Sweetgreen (DC staples; many locations)\\n\\nSit-down\\n- Old Ebbitt Grill (classic, near the White House)\\n- Founding Farmers (American, Foggy Bottom)\\n- Zaytinya (Mediterranean mezze, Penn Quarter)\\n- Daikaya (ramen, Chinatown)\\n- Unconventional Diner (modern American, Convention Center area)\\n- Le Diplomate (French bistro, 14th St/Logan Circle)\\n\\nNear the National Mall (if you\'re sightseeing)\\n- Mitsitam Cafe (Museum of the American Indian)\\n- Sweet Home Cafe (Museum of African American History & Culture)\\n- Pavilion Cafe (National Gallery Sculpture Garden)\\n\\nTell me your neighborhood, cuisine, and budget, and I\\u2019ll

Here we override the agent's default response format at runtime

In [20]:
class Restaurant(BaseModel):
    name: str
    cuisine: str

class TasteAtlas(BaseModel):
    restaurants: list[Restaurant]
    reasoning: str

molly(message='What are some places I would like? I tend to like Italian and Asian cuisines', response_format=TasteAtlas, chat='travel')


Load started on 04/16/2026 16:04:06
Loading file Agents.Message.Restaurant.cls as udl
Compiling class Agents.Message.Restaurant
Compiling routine Agents.Message.Restaurant.1
Load finished successfully.

Load started on 04/16/2026 16:04:06
Loading file Agents.Message.TasteAtlas.cls as udl
Compiling class Agents.Message.TasteAtlas
Compiling table Agents_Message.TasteAtlas
Compiling routine Agents.Message.TasteAtlas.1
Load finished successfully.


'{"restaurants": [{"name": "Osteria Morini", "cuisine": "Italian"}, {"name": "RPM Italian", "cuisine": "Italian"}, {"name": "Centrolina", "cuisine": "Italian"}, {"name": "Sfoglina", "cuisine": "Italian"}, {"name": "L\'Ardente", "cuisine": "Italian"}, {"name": "Filomena Ristorante", "cuisine": "Italian"}, {"name": "Officina", "cuisine": "Italian"}, {"name": "Daikaya", "cuisine": "Asian (Japanese - Ramen)"}, {"name": "Anju", "cuisine": "Asian (Korean)"}, {"name": "Thip Khao", "cuisine": "Asian (Lao)"}, {"name": "Maketto", "cuisine": "Asian (Cambodian/Taiwanese)"}, {"name": "Sushi Taro", "cuisine": "Asian (Japanese - Sushi)"}, {"name": "Rasika", "cuisine": "Asian (Indian)"}, {"name": "Tiger Fork", "cuisine": "Asian (Hong Kong)"}, {"name": "Pho 14", "cuisine": "Asian (Vietnamese)"}, {"name": "Tonari", "cuisine": "Italian-Japanese (Wafu Pasta)"}], "reasoning": "Since you like Italian and Asian, here are reliable Washington, DC lunch spots across both cuisines (mix of quick-casual and sit-do

Usage information can be fetched using the API for each Agent, Production or Chat. Productions can also be filtered by certain agents.

In [21]:
Chat('travel').usage()

'{"input_tokens": 10467, "output_tokens": 30992, "output_reasoning_tokens": 26496, "total_tokens": 41459}'

In [22]:
Production('AgentSpace').usage()

{'input_tokens': 17241,
 'output_tokens': 77990,
 'output_reasoning_tokens': 57984,
 'total_tokens': 95231}

In [23]:
molly.usage()

{'input_tokens': 17241,
 'output_tokens': 77990,
 'output_reasoning_tokens': 57984,
 'total_tokens': 95231}

In [24]:
Production('AgentSpace').usage(agents=[Agent('Molly')])

{'input_tokens': 17241,
 'output_tokens': 77990,
 'output_reasoning_tokens': 57984,
 'total_tokens': 95231}

Productions and Agents can be deleted using the `delete()` methods. This operation removes the Objectscript classes backing these objects as well.

In [25]:
Production('AgentSpace').delete()

Deleted production: User.AgentSpace

Deleting class Agents.REST.Dispatch.AgentSpaceCleaned up production-owned artifacts for: AgentSpace


In [26]:
Agent('Molly').delete()
try:
    molly('Hello')
except KeyError as e:
    print(e)


Deleting class Agents.Gateway.MollyService
Deleting class Agents.Process.Molly"No Agent found for 'Molly'"
